# autofollowdown — every command in 2 minutes ⚡

One `pip install`, then a single command line does it all: **compress · benchmark · pick the best · and run big LLMs on a *free* GPU**.

This notebook is a guided tour of the whole CLI. Every code cell below runs the **real** `autofollowdown` command and shows its **real** output — nothing faked.

Repo: https://github.com/peetwan/autofollowdown

## ⬇️ Install (one line)

Not on PyPI yet, so install straight from GitHub. On Colab this is the only setup you need:

```bash
!pip install "autofollowdown[examples] @ git+https://github.com/peetwan/autofollowdown"
```

After it installs, the `autofollowdown` command is on your PATH — so on Colab you can type `!autofollowdown info` directly. The cell below makes the package importable (installing from GitHub if needed) and defines a tiny `sh()` helper so this notebook's outputs render anywhere.

In [1]:
import os, sys, subprocess
try:
    import autofollowdown
    ROOT = os.path.dirname(os.path.dirname(autofollowdown.__file__))
except ImportError:
    ROOT = os.path.abspath('..')                      # running from a clone?
    if os.path.isdir(os.path.join(ROOT, 'autofollowdown')):
        sys.path.insert(0, ROOT)
    try:
        import autofollowdown
    except ImportError:                               # fresh machine / Colab
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'autofollowdown[examples] @ git+https://github.com/peetwan/autofollowdown'])
        import autofollowdown
        ROOT = os.path.dirname(os.path.dirname(autofollowdown.__file__))

ENV = {**os.environ, 'PYTHONPATH': ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')}

def sh(cmd):
    '''Run the real autofollowdown CLI and print its output (like a terminal).'''
    print('$ ' + cmd)
    parts = cmd.split()
    args = parts[1:] if parts and parts[0] == 'autofollowdown' else parts
    r = subprocess.run([sys.executable, '-m', 'autofollowdown.cli', *args],
                       capture_output=True, text=True, env=ENV)
    print((r.stdout or '').rstrip())
    if r.returncode:
        print(r.stderr.rstrip())

print('autofollowdown', autofollowdown.__version__, 'ready ✅')

autofollowdown 0.6.0 ready ✅


## 1️⃣  The whole command surface at a glance

`autofollowdown --help` lists everything it can do — one tool, every command below.

In [2]:
sh('autofollowdown --help')

$ autofollowdown --help
usage: autofollowdown [-h] [--version]
                      {compress,auto,recommend,diagnose,advise,gpu,info,benchmark-vision,benchmark-llm,autopick}
                      ...

Unified model compression (quantize/prune/distill) + real benchmarks.

positional arguments:
  {compress,auto,recommend,diagnose,advise,gpu,info,benchmark-vision,benchmark-llm,autopick}
    compress            ⭐ AUTO: profile → compress every way → benchmark →
                        pick → save (asks only if needed)
    auto                alias of `compress` (also accepts --model)
    recommend           find the best library for a model (esp. LLMs) and
                        explain why, with evidence
    diagnose            START HERE if you're stuck: "I can't run this model" →
                        exactly what to do
    advise              recommend WHICH technique (quantize/prune/distill) +
                        backend to use, and why
    gpu                 show your GPU +

## 2️⃣  `info` — what's under the hood

Shows the compression backends it can drive (native + the connected libraries) and the standard LLM benchmark tasks it knows about.

In [3]:
sh('autofollowdown info')

$ autofollowdown info


autofollowdown 0.6.0
Unified quantization · pruning · distillation, with real benchmarks.

Compression backends:
  • autofollowdown (native)                installed ✓
  • Microsoft NNI                          not installed (pip install nni)
  • llm-compressor (vLLM)                  not installed (pip install llmcompressor)
  • NVIDIA TensorRT Model Optimizer        not installed (pip install nvidia-modelopt)
  • torchao (PyTorch native)               not installed (pip install torchao)
  • bitsandbytes                           not installed (pip install bitsandbytes)
  • HQQ (Half-Quadratic Quantization)      not installed (pip install hqq)

LLM benchmark tasks (lm-eval-harness ids):
  perplexity             wikitext2, c4, ptb
  commonsense_zeroshot   arc_easy, arc_challenge, hellaswag, winogrande, piqa, openbookqa, boolq, lambada_openai
  knowledge              mmlu
  advanced_knowledge     mmlu_pro
  multilingual           mmlu_prox_en, mmlu_prox_lite_en
  reasoning              

## 3️⃣  `gpu` — run big LLMs on a *free* GPU 🆓

This is the new part. Quantizing an LLM normally needs a lot of VRAM, which makes it look expensive. `autofollowdown gpu` detects your GPU and prints a **memory-saving plan** based on the technique from llm-compressor's own docs — **sequential onloading**: only one slice of the model sits on the GPU at a time while the rest waits on CPU/disk. When VRAM is really tight it drops to onloading **one `Linear` at a time** (`sequential_targets="Linear"`).

The upshot: even a 70B model calibrates on a single **free 16 GB T4** — just a bit slower.

In [4]:
sh('autofollowdown gpu')   # detects this machine's GPU (CPU here)

$ autofollowdown gpu



GPU: none (CPU-only)
  Native INT8 runs here; the GPU backends need a free Colab/Kaggle T4.

Pass a model id to see its memory-saving plan, e.g.:
  autofollowdown gpu Qwen/Qwen3-0.6B


In [5]:
# The plan auto-escalates as the model outgrows a free 16 GB T4:
from autofollowdown import memory_plan
print(f"{'Model':>7} {'fp16':>7}  {'Strategy':<17} {'pipeline':<11} {'onload target':<14}")
print('-' * 60)
for name, p in [('0.6B', 0.6e9), ('7B', 7e9), ('70B', 70e9)]:
    pl = memory_plan(p, vram_gb=16)          # a free Colab/Kaggle T4 has 16 GB
    tgt = pl['sequential_targets'] or 'decoder layer'
    print(f"{name:>7} {p*2/1e9:>5.0f}GB  {pl['strategy']:<17} {pl['pipeline']:<11} {tgt:<14}")
print('\nAll three calibrate on ONE free 16 GB T4 — big models just onload one slice at a time.')

  Model    fp16  Strategy          pipeline    onload target 
------------------------------------------------------------


   0.6B     1GB  fits              basic       decoder layer 
     7B    14GB  sequential        sequential  decoder layer 
    70B   140GB  sequential        sequential  decoder layer 

All three calibrate on ONE free 16 GB T4 — big models just onload one slice at a time.


On a real Colab T4, point it at any model id to get that model's exact plan:

```bash
!autofollowdown gpu Qwen/Qwen3-0.6B
```

## 4️⃣  `recommend` — the best library for *your* model, and **why** 🧠

Give it a model and it ranks every backend for that model and explains the reasoning — so the pick is evidence-based, not a black box. Here it profiles a small CNN we save locally:

In [6]:
import torch, torch.nn as nn
cnn = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
                    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
                    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10))
torch.save(cnn.state_dict(), '/tmp/afd_demo_cnn.pt')
sh('autofollowdown recommend /tmp/afd_demo_cnn.pt --goal size')

$ autofollowdown recommend /tmp/afd_demo_cnn.pt --goal size



Model: /tmp/afd_demo_cnn.pt
  family=cnn · params=~0M · HF=False · CUDA=False

┌───┬─────────────────────────────────┬──────┬───────────────┬─────────────────────────────────────────┐
│ # │ Library                         │  Fit │ Status        │ Method                                  │
├───┼─────────────────────────────────┼──────┼───────────────┼─────────────────────────────────────────┤
│ 1 │ Microsoft NNI                   │ 0.96 │ not installed │ L1FilterPruner + ModelSpeedup           │
│ 2 │ NVIDIA TensorRT Model Optimizer │ 0.66 │ not installed │ INT8 PTQ                                │
│ 3 │ autofollowdown (native)         │ 0.60 │ runnable here │ unstructured-0.5 + int8-dynamic         │
│ 4 │ torchao (PyTorch native)        │ 0.56 │ not installed │ int8/int4 weight-only (+ torch.compile) │
└───┴─────────────────────────────────┴──────┴───────────────┴─────────────────────────────────────────┘

Why each library ranks where it does:
  • Microsoft NNI: Channel/filter pruning

For an LLM it ranks weight-only 4-bit (llm-compressor) at the top and tells you why — run this on Colab (it reads the model config, no weights downloaded):

```bash
!autofollowdown recommend Qwen/Qwen3-0.6B --goal accuracy
```

## 5️⃣  `diagnose` / `advise` — stuck? start from your problem 🩺

`advise` recommends *which* technique(s) to use (quantize / prune / distill) and **why**; `diagnose` answers the bigger question — *can I even run this, and what do I do?* Here `advise` plans the small CNN for speed:

In [7]:
sh('autofollowdown advise /tmp/afd_demo_cnn.pt --goal speed --can-retrain')

$ autofollowdown advise /tmp/afd_demo_cnn.pt --goal speed --can-retrain


Profiling /tmp/afd_demo_cnn.pt ...

Model: /tmp/afd_demo_cnn.pt  (family=cnn, goal=speed)

Recommended plan: structured pruning (channels/filters) → INT8 quantization

  1. structured pruning (channels/filters)  [via nni]
     why     : CNNs prune well structurally, and structured pruning gives a real latency win on ordinary hardware.
     expect  : ~1.3–2×, accuracy cost ~2–5% (recoverable by fine-tuning); speed real speedup on commodity hardware
  2. INT8 quantization  [via native]
     why     : the safe default — ~4× smaller with little accuracy loss and no retraining; portable to CPU.
     expect  : ~4× vs FP32 (≈2× vs FP16), accuracy cost usually <1–2%; speed up to ~2–4× on CPU

Best library for this model: Microsoft NNI   (runnable here: autofollowdown (native))

Watch out for:
  • fine-tune after pruning to recover accuracy
  • accuracy depends on calibration data matching real traffic

→ Verify on YOUR data before shipping: compress_and_benchmark(model)  (or  autofollowdown co

And `diagnose` — point it at any model plus your hardware (run on Colab):

```bash
!autofollowdown diagnose meta-llama/Llama-3.1-8B --problem won't-fit --vram 8
```

## 6️⃣  `compress` — the one-command headline ⭐

The easiest command: it compresses a model **every way**, **benchmarks** them all, and **recommends** the best size/quality trade-off. With no model it runs a fast offline demo on scikit-learn digits, so it works anywhere:

In [8]:
sh('autofollowdown compress --yes --epochs 4')

$ autofollowdown compress --yes --epochs 4



🤖 autofollowdown autopilot
   target : offline digits demo (no model given)
  🤖 What do you care about most? → balanced (auto)
   goal   : balanced   (override with --goal)

Training a baseline CNN on sklearn digits (offline demo) ...
Applying compression methods + benchmarking ...

=== Results ===

┌───────────────────────────────┬─────────┬─────────┬──────────┬────────┬───────┬──────────┬───────┬────────┬────────┐
│ Model                         │ Size MB │  Params │ Sparsity │ Lat ms │   Acc │ Fidelity │ Size× │ Speed× │   ΔAcc │
├───────────────────────────────┼─────────┼─────────┼──────────┼────────┼───────┼──────────┼───────┼────────┼────────┤
│ baseline                      │   1.078 │ 280,746 │     0.0% │   0.67 │ 90.4% │   100.0% │     — │      — │      — │
│ int8 dynamic                  │   0.304 │   9,568 │     0.0% │   1.05 │ 90.4% │   100.0% │ 3.55× │  0.64× │  +0.0% │
│ prune 50%                     │   1.078 │ 280,746 │    50.0% │   0.65 │ 91.6% │    98.9% │ 1.00× │  1

With a real model and `-o`, it saves the variant you pick:

```bash
!autofollowdown compress facebook/opt-125m -o small.pt        # take the recommended pick
!autofollowdown compress facebook/opt-125m -m 'int8 dynamic' -o small.pt   # or choose one
```

## 7️⃣  `autopick` — recommendations per model family

A quick cheat-sheet of which library wins for each kind of model:

In [9]:
sh('autofollowdown autopick')

$ autofollowdown autopick


Vision CNN
Model profile: ModelProfile(family=cnn, params=25,578, conv=True, transformer=False, hf=False, cuda=False)

Ranked compression backends:
 → 1. [0.60] autofollowdown (native): unstructured-0.5 + int8-dynamic (prune+quantize) — runnable
      Global magnitude pruning then portable INT8 — no GPU needed.
   2. [0.90] Microsoft NNI: L1FilterPruner + ModelSpeedup (structured-prune) — not installed
      Channel/filter pruning that ModelSpeedup turns into a genuinely smaller, faster model (real FLOP reduction).
      install: pip install nni
   3. [0.60] NVIDIA TensorRT Model Optimizer: INT8 PTQ (ptq) — not installed
      Calibrated PTQ (SmoothQuant/AWQ/NVFP4) via mtq.quantize, exportable to TensorRT. Requires an NVIDIA GPU.
      install: pip install nvidia-modelopt
   4. [0.50] torchao (PyTorch native): int8/int4 weight-only (+ torch.compile) (weight-only-quant) — not installed
      PyTorch-native int8/int4/fp8 with no calibration; pairs with torch.compile for fast inference an

There are also two ready benchmarks you can run directly:

```bash
!autofollowdown benchmark-vision        # offline CNN benchmark (sklearn digits)
!autofollowdown benchmark-llm           # WikiText-2 perplexity on facebook/opt-125m
```

## 🎉 That's the whole tool

From one install you get: `info` (what's inside) · `gpu` (a memory plan that runs big LLMs on a **free** GPU) · `recommend` (best library + why) · `compress` (compress → benchmark → pick in one command) · `autopick` · `benchmark-vision` / `benchmark-llm`.

Next: open the **[CPU demo](autofollowdown_cpu_demo.ipynb)** for the Python API with real numbers, or the **[Colab T4 notebook](autofollowdown_backends_colab.ipynb)** to see the connected libraries (NNI · llm-compressor · NVIDIA ModelOpt) run on a real GPU.